## Install the Google Gemini SDK

In [ ]:
!pip install -q google-genai

## Set up your Gemini API key

In [ ]:
import os
import getpass
from google import genai
from google.genai import types

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or getpass.getpass("Enter your Gemini API key: ")

client = genai.Client(api_key=GEMINI_API_KEY)

## Model

In [ ]:
MODEL = "gemini-2.5-flash"

## Generate text from text prompts

In [ ]:
for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents="Why is the sky blue?"
):
    print(chunk.text, end="")

#### Try your own prompts

- What are the biggest challenges facing the healthcare industry?
- What are the latest developments in the automotive industry?
- What are the biggest opportunities in retail industry?
- (Try your own prompts!)

## Use Gemini 2.5 Flash

The Gemini 2.5 Flash (`gemini-2.5-flash`) model is a fast, efficient multimodal model designed for natural language tasks, multi-turn conversations, code generation, and image/video understanding.

In [ ]:
prompt = """Create a numbered list of 10 items. Each item in the list should be a trend in the tech industry.

Each trend should be less than 5 words."""

for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents=prompt
):
    print(chunk.text, end="")

## Model parameters

Every prompt you send to the model includes parameter values that control how the model generates a response. You can experiment with different parameters to see how the results change.

In [ ]:
for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents="Why is the sky blue?",
    config=types.GenerateContentConfig(
        temperature=0.9,
        top_p=1.0,
        top_k=32,
        max_output_tokens=8192,
    )
):
    print(chunk.text, end="")

## Multi-turn chat

Gemini 2.5 Flash supports natural multi-turn conversations. The following examples show how the model responds during a multi-turn chat.

In [ ]:
chat = client.chats.create(model=MODEL)

prompt = """My name is Bappy. You are my personal assistant. My favorite movies are Lord of the Rings and Hobbit.

Suggest another movie I might like.
"""

response = chat.send_message(prompt)
print(response.text)

## Model parameters

Every prompt you send to the model includes parameter values that control how the model generates a response. The model can generate different results for different parameter values. You can experiment with different model parameters to see how the results change.

In [ ]:
response = chat.send_message("Are my favorite movies based on a book series?")
print(response.text)

### View chat history

In [ ]:
for message in chat.get_history():
    print(f"[{message.role}]")
    for part in message.parts:
        print(part.text)
    print()

## Use Gemini 2.5 Flash for multimodal tasks

Gemini 2.5 Flash natively supports multimodal inputs — text, images, and video — in a single model.

### Define helper functions

import urllib.request
import IPython.display
from PIL import Image as PIL_Image
import io


def display_image_from_url(url: str, max_width: int = 600) -> None:
    with urllib.request.urlopen(url) as response:
        image_bytes = response.read()
    img = PIL_Image.open(io.BytesIO(image_bytes))
    img.thumbnail((max_width, max_width))
    IPython.display.display(img)


def image_part_from_url(url: str, mime_type: str = "image/jpeg") -> types.Part:
    with urllib.request.urlopen(url) as response:
        image_bytes = response.read()
    return types.Part.from_bytes(data=image_bytes, mime_type=mime_type)

### Generate text from a local image file

In [ ]:
# Download a sample image
import urllib.request
urllib.request.urlretrieve(
    "https://storage.googleapis.com/cloud-samples-data/generative-ai/image/320px-Felis_catus-cat_on_snow.jpg",
    "image.jpg"
)

# Load image bytes and send to Gemini
with open("image.jpg", "rb") as f:
    image_bytes = f.read()

image_part = types.Part.from_bytes(data=image_bytes, mime_type="image/jpeg")

response = client.models.generate_content(
    model=MODEL,
    contents=["Describe this image.", image_part]
)
print(response.text)

### Generate text from an image URL

In [ ]:
import urllib.request
import IPython.display
from PIL import Image as PIL_Image
import io


def display_image_from_url(url: str, max_width: int = 600) -> None:
    with urllib.request.urlopen(url) as response:
        image_bytes = response.read()
    img = PIL_Image.open(io.BytesIO(image_bytes))
    img.thumbnail((max_width, max_width))
    IPython.display.display(img)


def image_part_from_url(url: str, mime_type: str = "image/jpeg") -> types.Part:
    with urllib.request.urlopen(url) as response:
        image_bytes = response.read()
    return types.Part.from_bytes(data=image_bytes, mime_type=mime_type)


image_url = "https://storage.googleapis.com/cloud-samples-data/generative-ai/image/boats.jpeg"

display_image_from_url(image_url)

image_part = image_part_from_url(image_url)
response = client.models.generate_content(
    model=MODEL,
    contents=["Describe the scene.", image_part]
)
print(response.text)

## Combining multiple images for few-shot prompting

In [ ]:
image1 = image_part_from_url("https://storage.googleapis.com/github-repo/img/gemini/intro/landmark1.jpg")
image2 = image_part_from_url("https://storage.googleapis.com/github-repo/img/gemini/intro/landmark2.jpg")
image3 = image_part_from_url("https://storage.googleapis.com/github-repo/img/gemini/intro/landmark3.jpg")

contents = [
    image1, '{"city": "London", "landmark": "Big Ben"}',
    image2, '{"city": "Paris", "landmark": "Eiffel Tower"}',
    image3, "Identify the city and landmark in this image as JSON."
]

response = client.models.generate_content(model=MODEL, contents=contents)
print(response.text)

### Generate text from a video file

Specify a publicly accessible video URL or a GCS URI. The supported MIME type for video includes `video/mp4`.

In [ ]:
video_url = "https://storage.googleapis.com/github-repo/img/gemini/multimodality_usecases_overview/pixel8.mp4"
IPython.display.Video(video_url, width=450)

prompt = """
Answer the following questions using the video only:
What is the profession of the main person?
What are the main features of the phone highlighted?
Which city was this recorded in?
Provide the answer as JSON.
"""

video_part = types.Part.from_uri(
    file_uri="gs://github-repo/img/gemini/multimodality_usecases_overview/pixel8.mp4",
    mime_type="video/mp4"
)

response = client.models.generate_content(
    model=MODEL,
    contents=[prompt, video_part]
)
print(response.text)